## Customer Support with the Handoffs (State Machine) Pattern
* Build a customer support agent whose behavior changes as it moves through a workflow. 

The agent will:
- Collect warranty information before proceeding.
- Classify issues as hardware or software.
- Provide a solution, or escalate to a human.
- Maintain conversation state across multiple turns.

Each "step" above is just a different configuration of the *same underlying agent*, selected
dynamically based on state -- not a separate agent instance.

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import AgentState
from typing_extensions import NotRequired
from typing import Literal
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage, HumanMessage
from langgraph.types import Command
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from rich import print
import sys
sys.path.append('..')
from utils.helper import print_messages

load_dotenv()

model = init_chat_model("openai:gpt-5-mini")
print("Model ready.")

Model ready.

## Walk through the workflow
<img src="../../assets/handoff_demo.png" width="500" height="500">

## 1. Define custom state

The `current_step` field is the core of the state machine: it determines which prompt and
tools are loaded on each turn.


In [4]:
class SupportState(AgentState):
    """State for the customer support workflow."""
    current_step: NotRequired[Literal["warranty_collector", "issue_classifier", "resolution_specialist"]]
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]
    issue_type: NotRequired[Literal["hardware", "software"]]

print("State schema defined.")


State schema defined.

## 2. Create tools that manage workflow state
**The key idea:** 
* Tools return `Command` objects that update **both** the data they collected *and* `current_step` 
* Which is what actually drives the workflow forward.

1. **record_warranty_status**
* `record_warranty_status` tool for handling the customer's warranty status and transitioning to issue classification.
* This sets `current_step` to `issue_classifier` because the next step after recording warranty status is to classify the issue.

2. **record_warranty_status**
* `record_issue_type` tool for handling the type of issue and transitioning to the `resolution_specialist`

In [7]:
@tool
def record_warranty_status(status: Literal["in_warranty", "out_of_warranty"], runtime: ToolRuntime) -> Command:
    """Record the customer's warranty status and transition to issue classification."""
    return Command(
        update={
            "messages": [
                ToolMessage(content=f"Warranty status recorded as: {status}", tool_call_id=runtime.tool_call_id)
            ],
            "warranty_status": status,
            "current_step": "issue_classifier"
        }
    )

@tool
def record_issue_type(issue_type: Literal["hardware", "software"], runtime: ToolRuntime) -> Command:
    """Record the type of issue and transition to resolution specialist."""
    return Command(
        update={
            "messages": [
                ToolMessage(content=f"Issue type recorded as: {issue_type}", tool_call_id=runtime.tool_call_id)
            ],
            "issue_type": issue_type,
            "current_step": "resolution_specialist",
        }
    )

@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the case to a human support specialist."""
    return f"Escalating to human support. Reason: {reason}"

@tool
def provide_solution(solution: str) -> str:
    """Provide a solution to the customer's issue."""
    return f"Solution provided: {solution}"

print("Workflow tools defined.")

Workflow tools defined.

## 3. Define each step's prompt and tools

A simple dictionary maps each step name to its prompt, its tools, and what state it requires
before it can run.


In [8]:
WARRANTY_COLLECTOR_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Warranty verification

At this step, you need to:
1. Greet the customer warmly
2. Ask if their device is under warranty
3. Use record_warranty_status to record their response and move to the next step

Be conversational and friendly. Don't ask multiple questions at once."""

ISSUE_CLASSIFIER_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Issue classification
CUSTOMER INFO: Warranty status is {warranty_status}

At this step, you need to:
1. Ask the customer to describe their issue
2. Determine if it's a hardware issue (physical damage, broken parts) or software issue (app crashes, performance)
3. Use record_issue_type to record the classification and move to the next step

If unclear, ask clarifying questions before classifying."""

RESOLUTION_SPECIALIST_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Resolution
CUSTOMER INFO: Warranty status is {warranty_status}, issue type is {issue_type}

At this step, you need to:
1. For SOFTWARE issues: provide troubleshooting steps using provide_solution
2. For HARDWARE issues:
   - If IN WARRANTY: explain warranty repair process using provide_solution
   - If OUT OF WARRANTY: escalate_to_human for paid repair options

Be specific and helpful in your solutions."""

STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
    },
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
    },
    "resolution_specialist": {
        "prompt": RESOLUTION_SPECIALIST_PROMPT,
        "tools": [provide_solution, escalate_to_human],
        "requires": ["warranty_status", "issue_type"],
    },
}
print("Step configuration defined -- add a new step just by adding a new dictionary entry.")


Step configuration defined -- add a new step just by adding a new dictionary entry.

## 4. Create step-based middleware
* We’ll use the `@wrap_model_call` decorator for a clean implementation:
* Middleware reads `current_step` from state on every model call
* Looks up the matching configuration, and overrides the prompt and tools for that turn only.

In [ ]:

@wrap_model_call
def apply_step_config(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Configure agent behavior based on the current step."""
    current_step = request.state.get("current_step", "warranty_collector")
    
    step_config = STEP_CONFIG[current_step]

    for required_step in step_config["requires"]:
        if request.state.get(required_step) is None:
            raise ValueError(f"{required_step} must be set before reaching {current_step}")
    
    system_prompt = step_config["prompt"].format(**request.state)
    request = request.override(system_prompt=system_prompt, tools=step_config["tools"])
    return handler(request)
print("Middleware defined.")

Middleware defined.

## 5. Create the agent

A checkpointer is required here too -- without it, `current_step` would be lost between
messages, and the workflow would restart from scratch on every turn.


In [ ]:
all_tools = [record_warranty_status, record_issue_type, provide_solution, escalate_to_human]
checkpointer = InMemorySaver()

support_agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=checkpointer,
)
print("Support agent created.")

Support agent created.

In [18]:
thread_id = 'user_12345'
config = {"configurable": {"thread_id": thread_id}}

print("=== Turn 1: Warranty Collection ===")
result = support_agent.invoke({"messages": [HumanMessage("Hi, I have an issue with my iPhone")]}, config)
print_messages(result)

=== Turn 1: Warranty Collection ===

HumanMessage: Hi, I have an issue with my iPhone

================================================================================

Agent: Hi — sorry you're having trouble with your iPhone, I'm here to help. Is your iPhone still under warranty 
(manufacturer warranty or AppleCare)?

================================================================================

In [19]:
print("=== Turn 2: Warranty Response ===")
result = support_agent.invoke({"messages": [HumanMessage("Yes, it's still under warranty")]}, config)
print_messages(result)

=== Turn 2: Warranty Response ===

HumanMessage: Hi, I have an issue with my iPhone

================================================================================

Agent: Hi — sorry you're having trouble with your iPhone, I'm here to help. Is your iPhone still under warranty 
(manufacturer warranty or AppleCare)?

================================================================================

...

================================================================================

ToolMessage: Warranty status recorded as: in_warranty

================================================================================

Agent: Thanks — since it's under warranty, I can help arrange next steps. First I need to understand the problem so
I can classify it as hardware or software.

Can you describe exactly what’s happening? Useful details:
- What are the symptoms (e.g., won’t turn on, screen unresponsive, cracked screen, no sound, camera not working, 
apps crash, very slow, stuck on Apple logo, update failed, battery drains quickly)?
- When did the issue start and did anything happen right before it (drop, liquid exposure, iOS update, app 
installed)?
- Is there any visible physical damage (cracks, dents, swelling, corrosion)?
- Does the phone power on and boot normally? Does the touchscreen respond?
- Are there error messages, codes, or specific apps that fail?
- iPhone model and iOS version (if you know them).

If you prefer, answer just the most relevant questions (e.g., “screen cracked” or “apps crash after update”) and 
I’ll classify it and move you to the right support path.

================================================================================

In [20]:
print(f"\nCurrent step: {result.get('current_step')}")

Current step: issue_classifier

In [21]:
print("=== Turn 3: Issue Description ===")
result = support_agent.invoke({"messages": [HumanMessage("The screen is physically cracked from dropping it")]}, config)
print_messages(result)

=== Turn 3: Issue Description ===

HumanMessage: Hi, I have an issue with my iPhone

================================================================================

Agent: Hi — sorry you're having trouble with your iPhone, I'm here to help. Is your iPhone still under warranty 
(manufacturer warranty or AppleCare)?

================================================================================

...

================================================================================

ToolMessage: Solution provided: Next steps to schedule repair:

1) Tell me:  - Your iPhone model (e.g., iPhone 12, 13 Pro)  - Country/region  - Whether you have AppleCare+

2) I will then:
- Estimate the likely repair fee (AppleCare+ fee or out-of-warranty estimate) for screen replacement, based on 
model/region.  
- Provide nearby Apple Store or AASP locations and links to schedule a Genius Bar appointment or mail-in repair 
options.  

If you want to proceed immediately, give me the three details above and I’ll find the fastest service option and 
help book it for you.

================================================================================

Agent: Thanks — since the screen is cracked and the device is in warranty, I can help arrange the repair. Do you 
have AppleCare+? Also tell me your iPhone model and which country/region you’re in and I’ll estimate the likely fee
and help schedule the nearest repair option.

================================================================================

In [22]:
print(f"\nCurrent step: {result.get('current_step')}")

Current step: resolution_specialist

In [23]:
print("=== Turn 4: Resolution ===")
result = support_agent.invoke({"messages": [HumanMessage("What should I do?")]}, config)
print_messages(result)

=== Turn 4: Resolution ===

HumanMessage: Hi, I have an issue with my iPhone

================================================================================

Agent: Hi — sorry you're having trouble with your iPhone, I'm here to help. Is your iPhone still under warranty 
(manufacturer warranty or AppleCare)?

================================================================================

...

================================================================================

ToolMessage: Solution provided: Immediate actions (do these now):
- Back up your iPhone right away (iCloud: Settings >  > iCloud > iCloud Backup > Back Up Now; or connect to a 
Mac/PC and Back Up Now in Finder/iTunes). Repairs can erase your device.
- If the glass is loose or sharp, power the phone off and avoid using it to prevent cuts or further internal 
damage. If you must move it, cover the screen with clear packing tape or a temporary screen protector.
- Remove any case, screen protector, and the SIM card if you plan to bring it in for service.

Prepare for service (what to have ready):
- Apple ID and password (technicians may ask you to disable Find My iPhone for certain repairs). Don’t disable 
until instructed if doing remote diagnostics; otherwise have credentials ready.
- Photo ID and proof of purchase (receipt or order email) if you have them.
- Device serial number (Settings > General > About) — helpful for checking warranty/coverage.

Warranty & coverage note:
- The standard Apple limited warranty does NOT cover accidental damage from drops. If you have AppleCare+ (or 
equivalent accidental-damage coverage), screen replacement is usually covered for a service fee. If you don’t have 
AppleCare+, you’ll receive an out-of-warranty repair quote even though the device’s limited warranty may still be 
active.

How to get it repaired (fastest options):
- Apple Store (Genius Bar): schedule via the Apple Support app or support.apple.com — in-person repairs are often 
same-day if parts are available.
- Apple Authorized Service Provider (AASP): often convenient and can use genuine Apple parts.
- Mail-in repair through Apple Support: Apple will send a prepaid box/label; this usually takes several days plus 
shipping.

What to expect at the appointment:
- Diagnostic check to confirm damage.
- If covered by AppleCare+, you’ll pay the applicable accidental-damage fee; if not, you’ll be given a quote and 
asked to approve charges before repair.
- Turnaround: same/next day for in-store if parts in stock; mail-in typically takes ~3–7 business days (varies by 
location/part availability).

How I can help you next:
- I can check whether you have AppleCare+ (if you provide your Apple ID email or device serial number) or estimate 
likely screen-replacement fees if you tell me your iPhone model and country/region.
- I can find nearby Apple Stores or AASPs and help schedule an appointment or start a mail-in request.

What I need from you to proceed: 1) Do you have AppleCare+? 2) Your iPhone model (e.g., iPhone 13 Pro) and 3) your 
country/region. Tell me those and I’ll estimate costs and find the fastest repair option or book an appointment for
you.

================================================================================

Agent: Do this now:
1) Back up your iPhone (iCloud or to a computer).  
2) If glass is loose/sharp, power it off and avoid using it — cover with clear tape if you must move it.  
3) Remove case/screen protector and SIM card; have your Apple ID/password, photo ID, and proof of purchase ready.

Next (repair options):
- If you have AppleCare+: screen replacement is usually covered for a service fee — schedule a Genius Bar 
appointment, visit an Apple Authorized Service Provider, or request mail-in service.  
- If you don’t have AppleCare+: you’ll receive an out-of-warranty repair quote; the same service options apply.

If you want, tell me: 1) Do you have AppleCare+? 2) iPhone model? 3) Country/region? I’ll estimate the likely fee 
and help find/schedule the fastest repair option.

================================================================================

In [24]:
print(f"\nCurrent step: {result.get('current_step')}")

Current step: resolution_specialist

## 7. Add flexibility: letting the customer go back to previous state
Let's allow correcting the warranty status or issue classification from the resolution step.
* If your workflow needs arbitrary transitions between most steps
* This pattern works best with a clear sequential progression plus occasional backward corrections, not a free-for-all.

**Note:**
* Every tool call the model makes needs a matching `ToolMessage` back in the state update
* The same way `record_warranty_status` and `record_issue_type` already return one above. 
* A tool that only returns `Command(update={"current_step": ...})` with no `ToolMessage` raises *"Expected to have a matching ToolMessage in Command.update... Every tool call... MUST have a corresponding ToolMessage"* the first time it's actually called. 
* So `go_back_to_warranty` and `go_back_to_classification` need a `ToolRuntime` parameter and a `ToolMessage` in their update, just like the other state-changing tools.

In [25]:
@tool
def go_back_to_warranty(runtime: ToolRuntime) -> Command:
    """Go back to warranty verification step."""
    return Command(
        update={
            "messages": [ToolMessage(content="Going back to warranty verification.", tool_call_id=runtime.tool_call_id)],
            "current_step": "warranty_collector",
        }
    )

@tool
def go_back_to_classification(runtime: ToolRuntime) -> Command:
    """Go back to issue classification step."""
    return Command(
        update={
            "messages": [ToolMessage(content="Going back to issue classification.", tool_call_id=runtime.tool_call_id)],
            "current_step": "issue_classifier",
        }
    )

STEP_CONFIG["resolution_specialist"]["tools"].extend([go_back_to_warranty, go_back_to_classification])

RESOLUTION_SPECIALIST_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Resolution
CUSTOMER INFO: Warranty status is {warranty_status}, issue type is {issue_type}

At this step, you need to:
1. For SOFTWARE issues: provide troubleshooting steps using provide_solution
2. For HARDWARE issues:
   - If IN WARRANTY: explain warranty repair process using provide_solution
   - If OUT OF WARRANTY: escalate_to_human for paid repair options

If the customer indicates any information was wrong, use:
- go_back_to_warranty to correct warranty status
- go_back_to_classification to correct issue type

Be specific and helpful in your solutions."""
STEP_CONFIG["resolution_specialist"]["prompt"] = RESOLUTION_SPECIALIST_PROMPT

all_tools = all_tools + [go_back_to_warranty, go_back_to_classification]
support_agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=checkpointer,
)

In [26]:

result = support_agent.invoke({"messages": [HumanMessage("Actually, I made a mistake - my device is out of warranty")]}, config)
print_messages(result)

HumanMessage: Hi, I have an issue with my iPhone

================================================================================

Agent: Hi — sorry you're having trouble with your iPhone, I'm here to help. Is your iPhone still under warranty 
(manufacturer warranty or AppleCare)?

================================================================================

HumanMessage: Yes, it's still under warranty

...

================================================================================

ToolMessage: Escalating to human support. Reason: Customer has an out-of-warranty hardware issue: iPhone with a 
physically cracked screen. Need a human specialist to provide paid repair options and estimates, including: - 
Official Apple out-of-warranty screen-replacement costs by model and country/region (or ranges) - AppleCare+ 
accidental-damage fees if applicable (customer may still have it but earlier corrected warranty to out_of_warranty)
- Nearby Apple Store / Apple Authorized Service Provider options and typical turnaround times - Mail-in repair 
options and estimated total time and costs - Guidance on data preservation and whether diagnostic/repair will erase
device - Alternative third-party repair options and tradeoffs (cost, parts quality, effect on future Apple service)
Customer has not yet provided iPhone model or country/region; please request those if needed. Customer reported the
device was previously thought to be in warranty but is actually out_of_warranty; issue_type is hardware (cracked 
screen). Please respond with paid repair options and next steps to get the device repaired.

================================================================================

Agent: Thanks for the update — since your device is out of warranty and it’s a hardware issue (cracked screen), 
I’ve escalated this to our specialist team so they can provide detailed paid-repair options and exact cost 
estimates.

Meanwhile, here are immediate next steps you can take and what to expect:

What to do now
- Back up your iPhone (iCloud or to a computer) — repairs may erase the device.  
- Power it off if the glass is loose/sharp and avoid using it to prevent cuts and further damage.  
- Remove any case/screen protector and the SIM card if you’re bringing it in.  
- Have photo ID and proof of purchase available (if you want to request a courtesy repair quote).  
- Note the device serial number (Settings > General > About) and the exact iPhone model.

What the human specialist will provide
- Official out-of-warranty repair costs (screen replacement) for your exact iPhone model and country/region.  
- Available repair channels: Apple Store/AASP in-person, mail-in repair, estimated turnaround times.  
- Whether the repair will require erasing the device and how data is handled.  
- Optional: third-party repair alternatives, typical pricing ranges, and tradeoffs (warranty implications, parts 
quality).

To speed things up, please tell me:
1) Your iPhone model (e.g., iPhone 13 Pro).  
2) Your country/region or ZIP/postal code so we can find nearby repair options.

Once you share those, I’ll pass them to the specialist and they’ll return with exact paid-repair options and 
estimates — or if you prefer, I can wait and then help you schedule a repair appointment once the specialist 
provides costs.

================================================================================

## 8. Managing message history as the workflow grows
* As the agent progresses through steps, the message list keeps growing. 
* `SummarizationMiddleware` compresses earlier messages once a token threshold is crossed, while keeping recent context:

    ```python
    from langchain.agents.middleware import SummarizationMiddleware

    support_agent = create_agent(
        model,
        tools=all_tools,
        state_schema=SupportState,
        middleware=[
            apply_step_config,
            SummarizationMiddleware(model="gpt-5-nano", trigger=("tokens", 4000), keep=("messages", 10)),
        ],
        checkpointer=InMemorySaver(),
    )
    ```

## Understanding state transitions

| Turn | `current_step` going in | What happens | `current_step` after |
|---|---|---|---|
| 1 | `warranty_collector` (default) | Agent asks about warranty | `warranty_collector` (unchanged until tool call) |
| 2 | `warranty_collector` | `record_warranty_status` called | `issue_classifier` |
| 3 | `issue_classifier` | `record_issue_type` called | `resolution_specialist` |
| 4 | `resolution_specialist` | `provide_solution` or `escalate_to_human` called | `resolution_specialist` (terminal) |

**The key insight: tools drive the workflow by updating `current_step`, and middleware
responds by applying the right configuration on the next turn.**

## Summary

| Point | Why it matters |
|---|---|
| One agent, many configurations | Simpler than spinning up a separate agent per step |
| `current_step` lives in state, not in code | The same middleware function handles every step |
| Tools return `Command` to drive transitions | State changes and workflow progression happen together |
| A checkpointer is mandatory | Without it, `current_step` resets every turn |
| `requires` validation catches a skipped step early | Fails loudly instead of silently misbehaving |

**When to use this pattern:** a task that flows through clear, mostly-sequential stages, where
behavior should change as the task progresses, and where the agent needs to converse directly
with the user at each stage. For centralized orchestration across unrelated domains instead,
see the Subagents notebook.
